# Cosine Similarity Outfit Recommendation

Customer answers 8 questions → One-Hot Encoding → Customer Vector → Cosine Similarity → Top 3 outfits.

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load the outfit dataset

In [2]:
df = pd.read_csv("outfits_annotated_draft.csv")
outfits = df[df["clothing_type"] != "Accessory"].copy().reset_index(drop=True)
outfits.head()

,image_id,image_file,style,clothing_type,color,color_combination,fit,outfit_type,trend_importance,occasion
0,1890,1890.jpg,Sporty,T-shirt,Black,Dark,Regular,Simple,Very important,Sports
1,2119,2119.jpg,Sporty,Pants,Black,Dark,Regular,Athletic,Very important,Sports
2,2122,2122.jpg,Sporty,Hoodie,Blue,Bright,Regular,Athletic,Very important,Sports
3,1892,1892.jpg,Casual,T-shirt,Red,Bright,Regular,Simple,Slightly,Everyday
4,1906,1906.jpg,Casual,T-shirt,Blue,Bright,Regular,Simple,Slightly,Everyday


## 2. Features used for recommendation

In [3]:
features = ["style", "clothing_type", "color", "color_combination", "fit", "outfit_type", "trend_importance", "occasion"]

## 3. Ask the customer 8 questions

In [ ]:
def ask_choice(question, options):
    print("\n" + question)
    for i, option in enumerate(options, start=1):
        print(f"{i}. {option}")
    while True:
        try:
            choice = int(input("Choose a number: "))
            if 1 <= choice <= len(options):
                return options[choice - 1]
        except ValueError:
            pass
        print("Please enter a valid number.")

customer = {
    "style": ask_choice("1) What is your preferred clothing style?", ["Casual", "Formal", "Sporty", "Streetwear"]),
    "clothing_type": ask_choice("2) What type of clothes do you prefer?", ["T-shirt", "Shirt", "Hoodie", "Polo", "Pants", "Jacket"]),
    "color": ask_choice("3) What is your favorite color?", ["Black", "White", "Blue", "Red", "Green", "Gray", "Brown", "Pink", "Purple"]),
    "color_combination": ask_choice("4) What color combination do you prefer?", ["Neutral", "Bright", "Dark", "Mixed"]),
    "fit": ask_choice("5) What fit do you prefer?", ["Slim", "Regular", "Oversized"]),
    "outfit_type": ask_choice("6) What do you usually wear?", ["Simple", "Layered", "Athletic", "Trendy"]),
    "trend_importance": ask_choice("7) How important are fashion trends to you?", ["Not important", "Slightly", "Very important"]),
    "occasion": ask_choice("8) What type of outfit do you prefer most?", ["Everyday", "University", "Sports", "Occasions"])
}
customer_df = pd.DataFrame([customer])
customer_df


1) What is your preferred clothing style?
1. Casual
2. Formal
3. Sporty
4. Streetwear


## 4. One-Hot Encoding

In [ ]:
combined = pd.concat([outfits[features], customer_df[features]], ignore_index=True)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
vectors = encoder.fit_transform(combined)

outfit_vectors = vectors[:-1]
customer_vector = vectors[-1].reshape(1, -1)

## 5. Calculate Cosine Similarity

In [ ]:
scores = cosine_similarity(customer_vector, outfit_vectors)[0]
outfits["similarity_score"] = scores

## 6. Recommend the Top 3 outfits

In [ ]:
top3 = outfits.sort_values("similarity_score", ascending=False).head(3)

print("=" * 55)
print("YOUR TOP 3 RECOMMENDED OUTFITS")
print("=" * 55)

for rank, (_, row) in enumerate(top3.iterrows(), start=1):
    print(f"\n#{rank}")
    print(f"Image: {row['image_file']}")
    print(f"Style: {row['style']}")
    print(f"Type: {row['clothing_type']}")
    print(f"Color: {row['color']}")
    print(f"Occasion: {row['occasion']}")
    print(f"Cosine similarity: {row['similarity_score']:.3f}")

## Conclusion

Cosine Similarity compares the customer's preference vector with available outfit vectors and recommends the three outfits with the highest similarity scores.